In [4]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
from datetime import datetime, timedelta

# Load base tables
claims_master_base = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_CLAIMS_MASTER_BASE.csv')
policy_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_POLICY_MASTER.csv')

# Ensure POLICY_NUMBER is unique in policy_master
policy_master_unique = policy_master.drop_duplicates(subset='POLICY_NUMBER')

# Add POLICY_CSL to claims_master_base for relevant POLICY_NUMBER
claims_master_base['POLICY_CSL'] = claims_master_base['POLICY_NUMBER'].map(policy_master_unique.set_index('POLICY_NUMBER')['POLICY_CSL'])

# Add POLICY_DEDUCTIBLE to claims_master_base for relevant POLICY_NUMBER
claims_master_base['DEDUCTIBLE'] = claims_master_base['POLICY_NUMBER'].map(policy_master_unique.set_index('POLICY_NUMBER')['POLICY_DEDUCTIBLE'])

# Initialize the progress bar
tqdm.pandas()

# Function to calculate TOTAL_CLAIM_AMOUNT
def calculate_total_claim_amount(row):
    csl = int(row['POLICY_CSL'].split('/')[0]) * 100000
    if row['LOSS_TYPE'] == 'Trivial Damage':
        return random.randint(150, 5000)
    elif row['LOSS_TYPE'] == 'Minor Damage':
        return random.randint(5001, 10000)
    elif row['LOSS_TYPE'] == 'Major Damage':
        return random.randint(10001, int(0.74 * csl))
    elif row['LOSS_TYPE'] == 'Total Loss':
        if row['INCIDENT_TYPE'] == 'Vehicle Theft':
            return csl
        else:
            return random.randint(int(0.75 * csl), csl)

# Function to calculate INJURY_CLAIM_AMOUNT
def calculate_injury_claim_amount(row):
    if row['INJURY_DAMAGE'] == 'No Injury Damage' or pd.isna(row['TOTAL_CLAIM_AMOUNT']):
        return np.nan
    else:
        return 0.25 * row['TOTAL_CLAIM_AMOUNT']

# Function to calculate PROPERTY_CLAIM_AMOUNT
def calculate_property_claim_amount(row):
    if row['PROPERTY_DAMAGE'] == 'No Property Damage' or pd.isna(row['TOTAL_CLAIM_AMOUNT']):
        return np.nan
    else:
        return 0.30 * row['TOTAL_CLAIM_AMOUNT']

# Function to calculate VEHICLE_CLAIM_AMOUNT
def calculate_vehicle_claim_amount(row):
    if pd.isna(row['TOTAL_CLAIM_AMOUNT']):
        return np.nan
    else:
        return row['TOTAL_CLAIM_AMOUNT'] - (row['INJURY_CLAIM_AMOUNT'] if not pd.isna(row['INJURY_CLAIM_AMOUNT']) else 0) - (row['PROPERTY_CLAIM_AMOUNT'] if not pd.isna(row['PROPERTY_CLAIM_AMOUNT']) else 0)

# Function to calculate TOTAL_PAID_CLAIM_AMOUNT
def calculate_total_paid_claim_amount(row):
    if pd.isna(row['TOTAL_CLAIM_AMOUNT']) or row['TOTAL_CLAIM_AMOUNT'] - row['DEDUCTIBLE'] <= 0:
        return 0
    elif random.random() <= 0.85:
        return row['TOTAL_CLAIM_AMOUNT'] - row['DEDUCTIBLE']
    else:
        return 0

# Function to determine CLAIM_STATUS
def determine_claim_status(row):
    if pd.isna(row['TOTAL_CLAIM_AMOUNT']) or row['TOTAL_CLAIM_AMOUNT'] - row['DEDUCTIBLE'] <= 0:
        return 'Within Deductible'
    elif row['TOTAL_PAID_CLAIM_AMOUNT'] == 0:
        return 'Rejected'
    else:
        return 'Paid'

# Function to generate random date within a range
def generate_random_date(start_date, end_date):
    delta = end_date - start_date
    random_days = random.randint(0, delta.days)
    return start_date + timedelta(days=random_days)

# Apply calculations and generate synthetic data with progress bar
claims_master_base['TOTAL_CLAIM_AMOUNT'] = claims_master_base.progress_apply(calculate_total_claim_amount, axis=1)
claims_master_base['INJURY_CLAIM_AMOUNT'] = claims_master_base.progress_apply(calculate_injury_claim_amount, axis=1)
claims_master_base['PROPERTY_CLAIM_AMOUNT'] = claims_master_base.progress_apply(calculate_property_claim_amount, axis=1)
claims_master_base['VEHICLE_CLAIM_AMOUNT'] = claims_master_base.progress_apply(calculate_vehicle_claim_amount, axis=1)
claims_master_base['TOTAL_PAID_CLAIM_AMOUNT'] = claims_master_base.progress_apply(calculate_total_paid_claim_amount, axis=1)
claims_master_base['CLAIM_STATUS'] = claims_master_base.progress_apply(determine_claim_status, axis=1)
claims_master_base['DOCS_SUBMISSION_DATE'] = claims_master_base.progress_apply(lambda row: generate_random_date(pd.to_datetime(row['CLAIM_REPORTING_DATE']), pd.to_datetime(row['CLAIM_REPORTING_DATE']) + timedelta(days=7)), axis=1)
claims_master_base['ACKNOWLEDGEMENT_DATE'] = claims_master_base.progress_apply(lambda row: generate_random_date(pd.to_datetime(row['DOCS_SUBMISSION_DATE']), pd.to_datetime(row['DOCS_SUBMISSION_DATE']) + timedelta(days=3)), axis=1)
claims_master_base['DECISION_DATE'] = claims_master_base.progress_apply(lambda row: generate_random_date(pd.to_datetime(row['ACKNOWLEDGEMENT_DATE']), pd.to_datetime(row['CLAIM_REPORTING_DATE']) + timedelta(days=35)), axis=1)
claims_master_base['CLAIM_PAYMENT_DATE'] = claims_master_base.progress_apply(lambda row: generate_random_date(pd.to_datetime(row['DECISION_DATE']), pd.to_datetime(row['CLAIM_REPORTING_DATE']) + timedelta(days=35)) if row['CLAIM_STATUS'] == 'Paid' else np.nan, axis=1)

# Save the generated data to a CSV file
claims_master_base.to_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_CLAIMS_MASTER.csv', index=False)

print("Synthetic data generation completed and saved to AUTO_INSURANCE_CLAIMS_MASTER.csv")



100%|██████████| 190649/190649 [01:39<00:00, 1911.94it/s]


Synthetic data generation completed and saved to AUTO_INSURANCE_CLAIMS_MASTER.csv


In [5]:
claims_master_base.head()

,CLAIMS_NUMBER,INCIDENT_DATE,POLICY_NUMBER,CLAIM_REPORTING_DATE,INCIDENT_TYPE,COLLISION_TYPE,LOSS_TYPE,AUTHORITIES_CONTACTED,INCIDENT_STATE,INCIDENT_CITY,...,TOTAL_CLAIM_AMOUNT,INJURY_CLAIM_AMOUNT,PROPERTY_CLAIM_AMOUNT,VEHICLE_CLAIM_AMOUNT,TOTAL_PAID_CLAIM_AMOUNT,CLAIM_STATUS,DOCS_SUBMISSION_DATE,ACKNOWLEDGEMENT_DATE,DECISION_DATE,CLAIM_PAYMENT_DATE
0,P44214256,2018-10-24,P71681514,2018-10-28,Parked Car,Side Collision,Trivial Damage,NaN,Texas,Vinton,...,904,NaN,NaN,904.00,654,Paid,2018-11-03,2018-11-03,2018-11-05,2018-11-24
1,P50717433,2018-03-28,P71656895,2018-04-09,Single Vehicle Collision,Front Collision,Trivial Damage,Ambulance,Texas,Lefors,...,690,172.50,NaN,517.50,0,Within Deductible,2018-04-11,2018-04-13,2018-05-10,NaT
2,P31918890,2015-12-17,P71661263,2015-12-24,Single Vehicle Collision,Rear Collision,Minor Damage,Police,Texas,Coffee City,...,9487,2371.75,NaN,7115.25,8487,Paid,2015-12-24,2015-12-24,2016-01-02,2016-01-11
3,P24921295,2019-01-31,P71641331,2019-02-13,Multi Vehicle Collision,Front Collision,Major Damage,Ambulance,Texas,Eidson Road,...,4522182,1130545.50,1356654.6,2034981.90,0,Rejected,2019-02-13,2019-02-13,2019-02-17,NaT
4,P69138812,2021-01-29,P71651819,2021-02-01,Single Vehicle Collision,Front Collision,Trivial Damage,Ambulance,Texas,Malakoff,...,4554,1138.50,1366.2,2049.30,2554,Paid,2021-02-08,2021-02-10,2021-03-07,2021-03-07


In [6]:
claims_master_base.shape

(190649, 27)

In [7]:
policy_master.shape

(953522, 24)